In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
# Color Map
color_map = {
    1: '#9e9e9eff',
    2: '#1a2e34ff',
    3: '#a89984ff', # Weeds
    4: '#d7a053', # Vinesa
    5: '#3f6367ff', # Trees
    6: 'pink',
    65: '#9e9e9eff'
}

In [5]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pdal
import geopandas as gpd
import laspy
from laspy import CopcReader
import shapely
from shapely.geometry import box
from scipy.spatial import cKDTree

# Custom functions
from vineyard_analysis.config import DATA_DIR
from vineyard_analysis.io.aoc import load_aoc, load_spacing, filter_aoc
from vineyard_analysis.io.parcels import load_parcels, asign_aoc_to_parcels
from vineyard_analysis.io.zones import load_zones


# Custom functions - in progress
from vineyard_analysis.analysis.clustering import cluster_points
from vineyard_analysis.analysis.row_analysis import find_row_orientation
from vineyard_analysis.lidar.lidar_file_urls import lidar_file_urls
from vineyard_analysis.lidar.download_all import download_all

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Load data
***
Parcels, AOC boundaries, and LiDAR tile zones

In [6]:
aoc = filter_aoc(load_aoc().merge(load_spacing(), on="PDOid", how="left"))
zones = load_zones()
parcels = asign_aoc_to_parcels(load_parcels(), aoc)

In [7]:
for index in range(len(parcels)):
    # Select individual parcel from dataframe
    plot = parcels.iloc[[index]]
    print(plot)

   index             IDU         AREA   SUP   UN    YEAR  LVSH  \
0      0  840490000D0295  3631.465269  0.37  3.0  2021.0   0.0   

                                            geometry  index_right  \
0  POLYGON ((859251.107 6342446.957, 859258.668 6...            3   

          PDOid  ...   Wine_Region Designation_Level min_row_spacing  \
0  PDO-FR-A0143  ...  Rhône Valley               Cru             NaN   

  max_row_spacing  min_plant_spacing max_plant_spacing  max_area  \
0             2.5                1.0               1.5       2.5   

                                              source  notes tier  
0  https://ec.europa.eu/geographical-indications-...    NaN    4  

[1 rows x 22 columns]
   index             IDU        AREA    SUP   UN    YEAR  LVSH  \
1      1  840490000D0296  912.809956  0.094  1.0  2021.0   0.0   

                                            geometry  index_right  \
1  POLYGON ((859309.399 6342550.218, 859311.447 6...            3   

          PDOid  

In [9]:
for index in range(len(parcels)):
    # Select individual parcel from dataframe
    plot = parcels.iloc[[index]]
    
    urls = lidar_file_urls(plot, zones)
    
    sel_geometry = plot.geometry.iloc[index]

    points = merge_in_memory(download_all(urls).values())
    
    las = pd.DataFrame(points, columns = ['X', 'Y','Z','Classification'])
    las_clip = las_clip.rename(columns={"X": "x", "Y": "y", "Z": "z"})

    mask = shapely.contains_xy(sel_geometry, las.x, las.y)

    # Native boolean indexing automatically preserves header, point format, and all dimensions
    las_clip = las[mask]

    centroids = cluster_points(
    las_clip,
    spacing=plot.min_plant_spacing,    # max distance for points to count as the same plant
    min_points=3)

    centroids_df = pd.DataFrame(centroids, columns=["x", "y", "z"])

    angle_deg, row_spacing = find_row_orientation(centroids_df)

    print(urls)

NameError: name 'merge_in_memory' is not defined

## Load and clip LiDAR
***
Pick a parcel of interest, dissolve any multi-row geometry, then clip the relevant LiDAR tiles to it.

In [ ]:
urls = lidar_file_urls(parcels, zones)

### idk but it works

In [ ]:
"""
In-memory lidar download + merge.

Downloads .laz/.copc.laz files into memory and merges them into a single
numpy structured array using PDAL. Nothing touches your project disk.
"""

import requests
import numpy as np
import pdal
import json
import tempfile
import os
from concurrent.futures import ThreadPoolExecutor


def download_all(urls, max_workers=6, timeout=60):
    """
    Download all URLs into memory in parallel.

    Returns
    -------
    dict {url: bytes} for successful downloads.
    Failed URLs are printed and skipped.
    """
    def _fetch(url):
        try:
            r = requests.get(url, timeout=timeout)
            r.raise_for_status()
            return url, r.content
        except requests.RequestException as e:
            print(f"  ✗ {url.split('/')[-1]}: {e}")
            return url, None

    results = {}
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        for url, data in ex.map(_fetch, urls):
            if data is not None:
                results[url] = data
                print(f"  ✓ {url.split('/')[-1]} ({len(data) / 1e6:.1f} MB)")
    return results


def merge_in_memory(laz_bytes_list):
    """
    Merge a list of LAZ byte blobs into a single numpy structured array.

    PDAL's readers.las needs a file path, so we write each blob to a
    NamedTemporaryFile (in /tmp, auto-deleted) just long enough for PDAL
    to read it. The merged points live entirely in memory afterward.

    Parameters
    ----------
    laz_bytes_list : iterable of bytes
        Raw .laz or .copc.laz file contents.

    Returns
    -------
    numpy.ndarray
        Structured array with all points from all inputs.
    """
    tmp_paths = []
    try:
        for blob in laz_bytes_list:
            f = tempfile.NamedTemporaryFile(suffix=".laz", delete=False)
            f.write(blob)
            f.close()
            tmp_paths.append(f.name)

        pipeline_def = [{"type": "readers.las", "filename": p} for p in tmp_paths]
        pipeline_def.append({"type": "filters.merge"})

        pipeline = pdal.Pipeline(json.dumps(pipeline_def))
        pipeline.execute()
        return pipeline.arrays[0]
    finally:
        for p in tmp_paths:
            try:
                os.unlink(p)
            except OSError:
                pass


urls = lidar_file_urls(parcels, zones)
print(f"Downloading {len(urls)} files...")
blobs = download_all(urls)

print(f"\nMerging {len(blobs)} files...")
points = merge_in_memory(blobs.values())

print(f"\n✓ Merged: {len(points):,} points, fields: {points.dtype.names}")
# `points` is now a numpy structured array — use it directly
# e.g. points['X'], points['Z'], points['Classification'], etc.

## Cluster vine points
***
Group LiDAR returns into individual vines and take each cluster centroid.

In [ ]:
centroids = cluster_points(
    las_clip,
    spacing=plot.min_plant_spacing,    # max distance for points to count as the same plant
    min_points=3,
)

centroids_df = pd.DataFrame(centroids, columns=["x", "y", "z"])

angle_deg, row_spacing = find_row_orientation(centroids_df)


In [ ]:
plot.min_plant_spacing

In [ ]:
# Get Parcel Bounds
buffered_gdf = plot.to_crs(epsg=2154)
buffered_gdf.to_crs(epsg=2154)
buffered_gdf['geometry'] = buffered_gdf.buffer(-0.75, resolution=64)
buffered_gdf = buffered_gdf[~buffered_gdf.is_empty]

x_min, y_min, x_max, y_max = plot.total_bounds

# Choose your interval spacing along x
interval = min_plant_spacing 
spacing = max_row_spacing

slope = np.tan(np.radians(angle_deg))

x_mid = np.median(x_coordinates)
y_mid = np.median(y_coordinates)

# Generate points along each line
all_points = []
   
    for i in seq_y:
        # Calculate corresponding y values using y = slope * (x - x_mid) + y_intercept
        y_vals = slope * (x_vals - x_mid) + i
    
        all_points.append(np.column_stack((x_vals, y_vals)))
    
elif angle_deg > 45:
    y_vals = np.arange(y_min, y_max, interval)
    
    phase_offset = x_mid % spacing
    
    seq_x = np.arange(x_min - spacing + phase_offset, 
                      x_max + spacing, 
                      spacing)

    for i in seq_x:
        x_vals = (1 / slope) * (y_vals - y_mid) + i
    
        all_points.append(np.column_stack((x_vals, y_vals)))
        
# Combine all points into one array
all_points = np.vstack(all_points)

df = pd.DataFrame(all_points, columns=['lon', 'lat'])

points_gdf = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df.lon, df.lat), crs=parcels_dissolved.crs)

filtered_points = gpd.sjoin(points_gdf, buffered_gdf, predicate='within')

# Plot to verify
fig, ax = plt.subplots(figsize=(10, 10))
ax.scatter(filtered_points.lon, filtered_points.lat, color='red', alpha = .5, s=5, zorder=5)

ax.scatter(x_coordinates, y_coordinates, color = '#3d5a64ff', s = 5)
#ax.scatter(las_clip.x, las_clip.y, color = '#3d5a64ff', s = 3)

parcels_dissolved.boundary.plot(ax=ax, color='black', linewidth=2)
plt.show()

## Compare actual vs expected
***
Match each expected plant to the nearest detected centroid; report the percent of expected positions that have a plant within `threshold`.

In [ ]:
expected_points = filtered_points[["lon", "lat"]].to_numpy()
actual_points = centroids_df[["x", "y"]].to_numpy()

threshold = 1.5
tree = cKDTree(actual_points)
matched = sum(1 for p in expected_points if tree.query_ball_point(p, r=threshold))

n_expected = len(expected_points)
print(f"Points With Nearby Plants: {matched}")
print(f"Total Points:              {n_expected}")
print(f"Area With Plants:          {(matched / n_expected) * 100:.2f}%")
print(f"Expected:                  {filtered_points.LVSH.iloc[0]}%")

In [ ]:
import json as _json
import os as _os
import tempfile
import traceback
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import requests


# ---- In-memory LiDAR helpers (defined here to shadow any module-level
# download_all that requires an output_dir argument) -------------------------

def download_all(urls, max_workers=6, timeout=60):
    """Download all URLs into memory in parallel.

    Returns a dict {url: bytes} for successful downloads. Failed URLs are
    printed and skipped.
    """
    def _fetch(url):
        try:
            r = requests.get(url, timeout=timeout)
            r.raise_for_status()
            return url, r.content
        except requests.RequestException as e:
            print(f"  x {url.split('/')[-1]}: {e}")
            return url, None

    results = {}
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        for url, data in ex.map(_fetch, urls):
            if data is not None:
                results[url] = data
                print(f"  v {url.split('/')[-1]} ({len(data) / 1e6:.1f} MB)")
    return results


def merge_in_memory(laz_bytes_list):
    """Merge a list of LAZ byte blobs into one numpy structured array via PDAL."""
    tmp_paths = []
    try:
        for blob in laz_bytes_list:
            f = tempfile.NamedTemporaryFile(suffix=".laz", delete=False)
            f.write(blob)
            f.close()
            tmp_paths.append(f.name)

        pipeline_def = [{"type": "readers.las", "filename": p} for p in tmp_paths]
        pipeline_def.append({"type": "filters.merge"})

        pipeline = pdal.Pipeline(_json.dumps(pipeline_def))
        pipeline.execute()
        return pipeline.arrays[0]
    finally:
        for p in tmp_paths:
            try:
                _os.unlink(p)
            except OSError:
                pass


# ---- Per-parcel pipeline ---------------------------------------------------

OUT_CSV = Path(DATA_DIR).parent / "plant_coverage.csv"
THRESHOLD = 1.5  # meters

# Resume support: skip parcels already in the CSV from a previous run.
if OUT_CSV.exists():
    done = set(pd.read_csv(OUT_CSV)["IDU"].astype(str))
else:
    done = set()
    pd.DataFrame(columns=["IDU", "n_expected", "n_matched", "pct_matched",
                          "expected_lvsh", "angle_deg", "row_spacing"]
                 ).to_csv(OUT_CSV, index=False)

for index in range(len(parcels)):
    # Select individual parcel from dataframe
    plot = parcels.iloc[[index]]
    idu = str(plot["IDU"].iloc[0])

    if idu in done:
        continue

    print(f"[{index + 1}/{len(parcels)}] IDU={idu}", flush=True)

    try:
        urls = lidar_file_urls(plot, zones)

        sel_geometry = plot.geometry.iloc[0]
        points = merge_in_memory(download_all(urls).values())

        las = pd.DataFrame(points, columns=["X", "Y", "Z", "Classification"])
        las = las.rename(columns={"X": "x", "Y": "y", "Z": "z"})
        mask = shapely.contains_xy(sel_geometry, las.x, las.y)
        # Native boolean indexing automatically preserves header, point format,
        # and all dimensions
        las_clip = las[mask]

        centroids = cluster_points(
            las_clip,
            spacing=plot.min_plant_spacing.iloc[0],  # max distance for points
                                                      # to count as the same plant
            min_points=3,
        )
        centroids_df = pd.DataFrame(centroids, columns=["x", "y", "z"])
        angle_deg, row_spacing = find_row_orientation(centroids_df)

        # ---- Get Parcel Bounds ----------------------------------------------
        buffered_gdf = plot.to_crs(epsg=2154)
        buffered_gdf.to_crs(epsg=2154)
        buffered_gdf["geometry"] = buffered_gdf.buffer(-0.75, resolution=64)
        buffered_gdf = buffered_gdf[~buffered_gdf.is_empty]

        x_min, y_min, x_max, y_max = plot.total_bounds

        # Choose your interval spacing along x
        interval = float(plot.min_plant_spacing.iloc[0])
        spacing = float(plot.max_row_spacing.iloc[0])

        slope = np.tan(np.radians(angle_deg))

        x_coordinates = centroids_df.x
        y_coordinates = centroids_df.y
        x_mid = np.median(x_coordinates)
        y_mid = np.median(y_coordinates)

        # Generate points along each line
        all_points = []

        if angle_deg <= 45:
            x_vals = np.arange(x_min, x_max, interval)
            phase_offset = y_mid % spacing
            seq_y = np.arange(y_min - spacing + phase_offset,
                              y_max + spacing,
                              spacing)

            for i in seq_y:
                # Calculate corresponding y values using
                # y = slope * (x - x_mid) + y_intercept
                y_vals = slope * (x_vals - x_mid) + i
                all_points.append(np.column_stack((x_vals, y_vals)))

        elif angle_deg > 45:
            y_vals = np.arange(y_min, y_max, interval)
            phase_offset = x_mid % spacing
            seq_x = np.arange(x_min - spacing + phase_offset,
                              x_max + spacing,
                              spacing)

            for i in seq_x:
                x_vals = (1 / slope) * (y_vals - y_mid) + i
                all_points.append(np.column_stack((x_vals, y_vals)))

        # Combine all points into one array
        all_points = np.vstack(all_points)

        df = pd.DataFrame(all_points, columns=["lon", "lat"])

        points_gdf = gpd.GeoDataFrame(
            df, geometry=gpd.points_from_xy(df.lon, df.lat), crs=plot.crs)

        filtered_points = gpd.sjoin(points_gdf, buffered_gdf, predicate="within")

        # Plot to verify
        fig, ax = plt.subplots(figsize=(10, 10))
        ax.scatter(filtered_points.lon, filtered_points.lat,
                   color="red", alpha=.5, s=5, zorder=5)
        ax.scatter(x_coordinates, y_coordinates, color="#3d5a64ff", s=5)
        # ax.scatter(las_clip.x, las_clip.y, color = "#3d5a64ff", s = 3)
        plot.boundary.plot(ax=ax, color="black", linewidth=2)
        ax.set_title(f"IDU {idu}")
        plt.show()
        plt.close(fig)

        # ---- Compare actual vs expected ------------------------------------
        expected_points = filtered_points[["lon", "lat"]].to_numpy()
        actual_points = centroids_df[["x", "y"]].to_numpy()

        tree = cKDTree(actual_points)
        matched = sum(1 for p in expected_points
                      if tree.query_ball_point(p, r=THRESHOLD))

        n_expected = len(expected_points)
        pct = (matched / n_expected) * 100 if n_expected else float("nan")
        expected_lvsh = (
            float(filtered_points.LVSH.iloc[0])
            if "LVSH" in filtered_points.columns and len(filtered_points)
            else float("nan")
        )

        print(f"Points With Nearby Plants: {matched}")
        print(f"Total Points:              {n_expected}")
        print(f"Area With Plants:          {pct:.2f}%")
        print(f"Expected:                  {expected_lvsh}%")

        pd.DataFrame([{
            "IDU": idu,
            "n_expected": n_expected,
            "n_matched": matched,
            "pct_matched": round(pct, 2),
            "expected_lvsh": expected_lvsh,
            "angle_deg": round(float(angle_deg), 3),
            "row_spacing": round(float(row_spacing), 3),
        }]).to_csv(OUT_CSV, mode="a", header=False, index=False)
        done.add(idu)

    except Exception as e:
        print(f"    !! skipped IDU={idu}: {e}", flush=True)
        traceback.print_exc()
        continue

print(f"\nDone. Results in {OUT_CSV}")
